This is the reproduction notebook for the main PCSS + MiSS experiment for `Qwen3-4B-Base`.

This specific notebook was tested on this docker image on a H100 SXM: `vastai/pytorch:2.10.0-cuda-13.0.2-py312-24.04-2026-03-26`

It takes about ~6-8 minutes to train on a H100 / H200. You can use other hardware for sure, just make sure to switch `attn_implementation` accordingly (e.g., to 'sdpa')

If you are using the same docker image and hopper hardware (with cu130 compatibility), you can just run all cells. When you're connecting to the jupyter server, select `main venv`

##### Installation

Here we install `uv` and prerequisites:

In [2]:
%pip install uv
!/venv/main/bin/python -m uv pip install transformers==5.8.1 trl==1.4.0 peft==0.19.1 bitsandbytes==0.49.2 kernels==0.14.1
!/venv/main/bin/python -m uv pip install git+https://github.com/linkedin/Liger-Kernel.git@30b8486a2bd48dff97f22c5f0c88520b8825cb35

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 70.1 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.13 environment at: /venv/main
Resolved 79 packages in 663ms                                        
Prepared 31 packages in 1.07s                                            
Uninstalled 3 packages in 132ms
Installed 31 packages in 1.06s                              
 + accelerate==1.14.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + attrs==26.1.0
 + bitsandbytes==0.49.2
 + charset-normalizer==3.5.1
 - click==8.3.1
 + click==8.5.0
 + datasets==5.0.1
 + dill==0.4.1
 + frozenlist==1.8.0
 - hf-xet==1.4.2
 + hf-xet==1.6.0
 - huggingface-hub==1.8.0
 + huggingface-hub==1.30.0
 + kernels==0.14.1
 + kernels-data==0.16.1
 + multidict==6.7.1
 + multiprocess==0.70.19
 + pandas==3.0.5
 + peft==0.19.1
 + propcache==0.5.2
 + pyarrow==25.0.1
 + regex==2026.9.3
 + requests==2.34.2
 + safetensors==0.8.0
 + token

In case that something goes wrong, make sure `%pip freeze` matches the following output:

In [3]:
%pip freeze

accelerate==1.14.0
aiohappyeyeballs==2.7.1
aiohttp==3.14.3
aiosignal==1.4.0
annotated-doc==0.0.4
anyio==4.13.0
asttokens==3.0.1
attrs==26.1.0
bitsandbytes==0.49.2
certifi==2026.2.25
charset-normalizer==3.5.1
click==8.5.0
comm==0.2.3
cuda-bindings==13.0.3
cuda-pathfinder==1.5.0
datasets==5.0.1
debugpy==1.8.20
decorator==5.2.1
dill==0.4.1
executing==2.2.1
filelock==3.25.2
frozenlist==1.8.0
fsspec==2026.2.0
h11==0.16.0
hf-xet==1.6.0
httpcore==1.0.9
httpx==0.28.1
huggingface_hub==1.30.0
idna==3.11
ipykernel==7.2.0
ipython==9.11.0
ipython_pygments_lexers==1.1.1
ipywidgets==8.1.8
jedi==0.19.2
Jinja2==3.1.6
jupyter_client==8.8.0
jupyter_core==5.9.1
jupyterlab_widgets==3.0.16
kernels==0.14.1
kernels-data==0.16.1
liger_kernel @ git+https://github.com/linkedin/Liger-Kernel.git@30b8486a2bd48dff97f22c5f0c88520b8825cb35
markdown-it-py==4.0.0
MarkupSafe==3.0.3
matplotlib-inline==0.2.1
mdurl==0.1.2
mpmath==1.3.0
multidict==6.7.1
multiprocess==0.70.19
nest-asyncio==1.6.0
networkx==3.6.1
numpy==2.4.3
n

In [4]:
import os, torch, copy
from transformers import AutoModelForCausalLM, AutoTokenizer
from liger_kernel.transformers import apply_liger_kernel_to_qwen3
from typing import Optional, List
import torch
import torch.nn as nn

apply_liger_kernel_to_qwen3() # apply here so we benefit during inference

model = AutoModelForCausalLM.from_pretrained(
    # This is unsloth's mirror. They set pad_token to `<|vision_pad|>` etc
    # In my very old experiments, I used unsloth's mirror for this model and I've kept using it to this day
    # You can use `Qwen/Qwen3-4B-Base`, it doesn't matter much
    "unsloth/Qwen3-4B-Base",
    # Use for hopper
    attn_implementation = "kernels-community/flash-attn3@v1",
    #attn_implementation = "sdpa",
    dtype = torch.bfloat16,
    use_cache = False,
    device_map = "cuda"
)
tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen3-4B-Base")
tokenizer.chat_template = \
    "{% if messages[0]['role'] == 'system' %}"\
        "{{ raise_exception('Custom system message is not supported.') }}"\
    "{% endif %}"\
    "{% if messages|length > 2 %}"\
        "{{ raise_exception('Multi turn chat is not natively supported.') }}"\
    "{% endif %}"\
    "{% if messages[0]['role'] != 'user' %}"\
        "{{ raise_exception('First message is expected to be the user input.') }}"\
    "{% endif %}"\
    "{% if messages|length == 2 and messages[1]['role'] != 'assistant' %}"\
        "{{ raise_exception('Second message is expected to be the model response.') }}"\
    "{% endif %}"\
    "{{ bos_token }}"\
    "{{ '**User Input**\n\n' }}"\
    "{{ messages[0]['content'] + '\n\n' }}"\
    "{% if messages|length == 2 and messages[1]['role'] == 'assistant' %}"\
        "{{ '**AI Model Response**\n\n' }}"\
        "{{ messages[1]['content'] + eos_token }}"\
    "{% elif add_generation_prompt %}"\
        "{{ '**AI Model Response**\n\n' }}"\
    "{% endif %}"

config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

##### Dataset Processing

In [5]:
from datasets import Dataset, DatasetDict, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from typing import Union

def create_pcss_dataset(
    dataset: Union[Dataset, DatasetDict],
    tokenizer: AutoTokenizer,
    ref_model: nn.Module
) -> Union[Dataset, DatasetDict]:
    ref_model.eval()

    def _process_single_example(example: dict) -> dict:
        prompt_str = tokenizer.apply_chat_template(
            [
                {
                    "role": "user",
                    "content": example["prompt"]
                }
            ], 
            add_generation_prompt=True, 
            tokenize=False
        )
        full_str = prompt_str + example["completion"] + tokenizer.eos_token
        
        prompt_tokens = tokenizer(prompt_str, return_tensors="pt")
        full_tokens = tokenizer(full_str, return_tensors="pt")
        prompt_len = prompt_tokens.input_ids.shape[1]

        labels = full_tokens.input_ids.clone()
        labels[:, :prompt_len] = -100

        inputs_for_ref = {
            "input_ids": full_tokens.input_ids.to(ref_model.device),
            "attention_mask": full_tokens.attention_mask.to(ref_model.device),
            "labels": labels.to(ref_model.device),
        }

        with torch.no_grad():
            # note: skip_logits is an option for liger kernel
            outputs = ref_model(skip_logits=True, **inputs_for_ref)
            l_ref = outputs.loss.cpu().item()

        return {
            "input_ids": full_tokens.input_ids.squeeze(0).tolist(),
            "attention_mask": full_tokens.attention_mask.squeeze(0).tolist(),
            "labels": labels.squeeze(0).tolist(),
            "l_ref": l_ref,
        }

    if isinstance(dataset, DatasetDict):
        processed_splits = {}
        for split_name, split_dataset in dataset.items():
            processed_examples = [
                _process_single_example(ex)
                for ex in tqdm(split_dataset, desc=f"Processing split '{split_name}'")
            ]
            processed_splits[split_name] = Dataset.from_list(processed_examples)
        return DatasetDict(processed_splits)
    
    elif isinstance(dataset, Dataset):
        processed_examples = [
            _process_single_example(ex)
            for ex in tqdm(dataset, desc="Processing dataset")
        ]
        return Dataset.from_list(processed_examples)
    
    else:
        raise TypeError(f"Input must be a Dataset or DatasetDict, but got {type(dataset)}")

dataset = load_dataset("tamewild/zebra_100")

dataset = dataset.map(lambda example: {
    "prompt": example["conversation"][0]["content"],
    "completion": example["conversation"][1]["content"]
})

dataset = create_pcss_dataset(dataset, tokenizer, model)

README.md:   0%|          | 0.00/447 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.55MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.99MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/77 [00:00<?, ? examples/s]

Processing split 'train':   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Processing split 'test':   0%|          | 0/77 [00:00<?, ?it/s]

##### Dataset Inspection

I recommend using HF's dataset inspector on the website, it's convenient. But here you can inspect the first entry.

See first entry without prompt masked:

In [6]:
print(tokenizer.decode(dataset['train'][0]['input_ids']))

**User Input**

There are 5 houses in a row, numbered from 1 through 5 from left to right. Each house is home to one person and each resident has a unique attribute from the following characteristics:

Preferred Furniture Style: Bohemian, Industrial, Modern, Rustic, Traditional
Names: Frank, Charlie, Bob, Eve, David
Type of Car: Minivan, Hatchback, SUV, Sports Car, Truck
Favorite Cooking Method: Grilling, Sous Vide, Air Frying, Baking, Slow Cooking
Favorite Store: Online Craft Store, Target, Amazon, Local Bookstore, Specialty Coffee Shop

1. Charlie is somewhere to the left of the person who prefers modern furniture.
2. There is 1 house between Charlie and the rustic furniture lover.
3. The hatchback owner is next to the person whose favorite store is Amazon.
4. The sous vide enthusiast is not the local bookstore supporter.
5. Frank is somewhere to the left of the person who loves baking.
6. The SUV owner is next to Eve.
7. The truck driver is directly right of the sports car enthusias

See first entry with prompt masked:

In [7]:
def print_unmasked_part(entry):
    unmasked = list(filter(lambda label: label != -100, entry["labels"]))
    print(tokenizer.decode(unmasked))

print_unmasked_part(dataset['train'][0])

We are given a logic puzzle with 5 houses in a row, numbered 1 to 5 from left to right. Each house has a unique combination of:

- **Furniture Style**: Bohemian, Industrial, Modern, Rustic, Traditional  
- **Names**: Frank, Charlie, Bob, Eve, David  
- **Car**: Minivan, Hatchback, SUV, Sports Car, Truck  
- **Cooking Method**: Grilling, Sous Vide, Air Frying, Baking, Slow Cooking  
- **Favorite Store**: Online Craft Store, Target, Amazon, Local Bookstore, Specialty Coffee Shop  

We are given 19 clues. We'll solve this step by step using deduction.

---

### Step 1: Initialize the grid

We'll represent the houses as positions 1 to 5:

| House # | 1 | 2 | 3 | 4 | 5 |
|--------|---|---|---|---|---|
| Name   |   |   |   |   |   |
| Furniture |   |   |   |   |   |
| Car    |   |   |   |   |   |
| Cooking |   |   |   |   |   |
| Store  |   |   |   |   |   |

---

### Step 2: Direct Clues

Let’s go through the clues one by one.

#### Clue 17: The online craft store shopper is in the first ho

##### Training

In [8]:
from peft import get_peft_model, MissConfig

model = get_peft_model(
    model,
    MissConfig(
        r = 512,
        target_modules = "all-linear",
        bias = "none",
        modules_to_save = None,
        task_type="CAUSAL_LM",
        init_weights = True # "MiSS efficience and balance"
    ),
    autocast_adapter_dtype = False # important: keep in bf16
)

model.print_trainable_parameters()

trainable params: 566,231,040 || all params: 4,588,699,136 || trainable%: 12.3397


In [9]:
from transformers import Trainer, TrainingArguments, DefaultDataCollator
from typing import Dict, Union, Any, Optional

class PCSSTrainer(Trainer):
    def __init__(self, *args, beta, peak_scale, global_l_std, **kwargs):
        super().__init__(*args, **kwargs)
        self.beta = beta
        self.peak_scale = peak_scale
        self.global_l_std = global_l_std

    def compute_loss(
        self,
        model: nn.Module,
        inputs: Dict[str, Union[torch.Tensor, Any]],
        return_outputs: bool = False,
        num_items_in_batch: Optional[torch.Tensor] = None,
    ):
        l_ref = inputs.pop("l_ref")
        outputs = model(skip_logits=True, **inputs)
        l_sft = outputs.loss

        mode = "train" if self.model.training else "eval"

        if mode == "train":
            with torch.no_grad():
                advantage = (l_ref - l_sft) / self.global_l_std
                sigmoid_input = self.beta * advantage
                sigmoid_prime = torch.sigmoid(sigmoid_input) * (1 - torch.sigmoid(sigmoid_input))
                scale = self.peak_scale * (4.0 * sigmoid_prime)

            final_loss = (l_sft * scale.detach()).squeeze()

            return (final_loss, outputs) if return_outputs else final_loss
        else:
            # During evaluation, return the standard, unscaled SFT loss
            return (l_sft, outputs) if return_outputs else l_sft

In [10]:
from transformers import TrainingArguments, default_data_collator
import numpy as np

trainer = PCSSTrainer(
    model = model,
    train_dataset = dataset['train'],
    eval_dataset = dataset['test'],
    data_collator = default_data_collator,
    beta=0.65,
    peak_scale=5,
    global_l_std=np.std(dataset['train']['l_ref']),
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        per_device_eval_batch_size = 1,
        gradient_accumulation_steps = 1,
        warmup_steps = 100, # fixed at 1 epoch
        num_train_epochs = 5,
        learning_rate = 5e-6,
        fp16 = False,
        bf16 = True,
        logging_strategy = "no",
        optim = "adamw_8bit",
        adam_beta2 = 0.99994, # scale optimizer memory. according to paper
        weight_decay = 0.01,
        lr_scheduler_type = "constant_with_warmup",
        seed = 3407,
        output_dir = "workspace",
        save_strategy = "steps",
        save_steps = 500,
        eval_strategy = "epoch",
        gradient_checkpointing = False, # highly recommended for h100, h200, etc
        gradient_checkpointing_kwargs = {"use_reentrant": False},
        use_liger_kernel = False,
        prediction_loss_only = True,
        remove_unused_columns = False
    )
)

In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.385898
2,No log,0.346497
3,No log,0.339905
4,No log,0.344286
5,No log,0.359928


TrainOutput(global_step=500, training_loss=0.5283584594726562, metrics={'train_runtime': 347.0258, 'train_samples_per_second': 1.441, 'train_steps_per_second': 1.441, 'total_flos': 1.0399693339436544e+17, 'train_loss': 0.5283584594726562, 'epoch': 5.0})

In [12]:
import gc
from peft import PeftModel

# free vram
del model, trainer
torch.cuda.empty_cache()
gc.collect()

# merge with `autocast_adapter_dtype=True` (fp32)
model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Qwen3-4B-Base",
    attn_implementation = "sdpa",
    dtype = torch.bfloat16,
    device_map = "cuda"
)
model = PeftModel.from_pretrained(
    model,
    "/workspace/checkpoint-500",
    autocast_adapter_dtype = True
)
model = model.merge_and_unload()

# save locally, you can optionally upload to HF (look it up) but please avoid polluting HF with essentially the same model trained in a few minutes
model.save_pretrained("./merged")
tokenizer.save_pretrained("./merged")

# free vram again
del model
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

##### MATH-500 Evaluation using vLLM

In [13]:
# NOTE: This is for cu130, make sure your device supports it. Also pin fastapi because of https://github.com/vllm-project/vllm/issues/45597
!/venv/main/bin/python -m uv pip install vllm==0.19.1 fastapi==0.136.3 transformers==5.8.1 --extra-index-url https://wheels.vllm.ai/0.19.1/cu130
!/venv/main/bin/python -m uv pip install math-verify[antlr4_13_2]==0.9.0 #pinned version of hf math-verify

Using Python 3.12.13 environment at: /venv/main
Resolved 180 packages in 2.95s                                       
Prepared 112 packages in 4.62s                                           
Uninstalled 2 packages in 1.13s
Installed 112 packages in 7.39s6.6                          
 + agent-detector==2.0.0
 + annotated-types==0.8.0
 + anthropic==1.4.0
 + apache-tvm-ffi==0.1.13.post3
 + astor==0.8.1
 + blake3==1.0.9
 + cachetools==7.1.8
 + cbor2==6.1.4
 + cffi==2.1.1
 + cloudpickle==3.1.2
 + compressed-tensors==0.15.0.1
 + cryptography==50.0.1
 + cuda-python==13.0.3
 + depyf==0.20.0
 + detect-installer==0.2.1
 + diskcache==5.6.3
 + distro==1.9.0
 + dnspython==2.8.0
 + docstring-parser==0.18.0
 + einops==0.8.2
 + email-validator==2.3.0
 + fastapi==0.136.3
 + fastapi-cli==0.0.32
 + fastapi-cloud-cli==0.24.0
 + fastar==0.12.0
 + flashinfer-cubin==0.6.6
 + flashinfer-python==0.6.6
 + gguf==0.19.0
 + googleapis-common-protos==1.75.3
 + grpcio==1.83.1
 + httpcore2==2.3.0
 + httptools==0.8.0

In [14]:
from datasets import load_dataset
from vllm import LLM, SamplingParams
from math_verify import parse, verify, LatexExtractionConfig, ExprExtractionConfig

llm = LLM(
    model = "./merged",
    max_model_len = 12_000,
    gpu_memory_utilization = 0.85,
    generation_config = "vllm",
    disable_log_stats = False
)

def extract_last_box(s: str) -> str:
    tag = "\\boxed{"
    tag_start = s.rfind(tag)
    if tag_start == -1:
        return ""

    content = s[tag_start + len(tag):]
    depth = 1
    for i, char in enumerate(content):
        if char == '{':
            depth += 1
        elif char == '}':
            depth -= 1
            if depth == 0:
                return content[:i].strip()

    return ""

def check_math500(ground_truth, response) -> bool:
    boxed_content = extract_last_box(response)
    if not boxed_content:
        return False

    boxed_response = f"\\boxed{{{boxed_content}}}"

    # Ensure ground truth is parsable by math_verify
    if "\\boxed" not in ground_truth:
        ground_truth = f"\\boxed{{{ground_truth}}}"

    gold = parse(
        ground_truth,
        # https://github.com/huggingface/Math-Verify/blob/ba3d3aaff23b3f4cac7a14672b4f6e293d97c98b/src/math_verify/tasks.py#L219
        [LatexExtractionConfig(boxed_match_priority=0)]
    )
    resp = parse(
        boxed_response,
        # https://github.com/huggingface/Math-Verify/blob/ba3d3aaff23b3f4cac7a14672b4f6e293d97c98b/src/math_verify/tasks.py#L221
        [
            LatexExtractionConfig(boxed_match_priority=0),
            ExprExtractionConfig()
        ]
    )
    return verify(gold, resp)

def benchmark_math500(llm):
    math500_benchmark = load_dataset("HuggingFaceH4/MATH-500")['test']

    extra = r"Please reason step by step, and put your final answer within \boxed{}"

    math500_benchmark = math500_benchmark.map(lambda example: {
        "problem": example["problem"].strip() + f"\n\n{extra}"
    })

    prompts = [[{"role": "user", "content": question["problem"]}] for question in math500_benchmark]

    prompt_outputs = llm.chat(prompts, SamplingParams(temperature=0.0, max_tokens=11_000), use_tqdm=True)

    correct = 0

    for solution, prompt_output in zip(math500_benchmark['solution'], prompt_outputs):
        if check_math500(solution, prompt_output.outputs[0].text):
            correct += 1

    print(f"Correct answers: {correct}")
    print(f"Estimated pass@1: {correct / len(math500_benchmark['problem'])}")

benchmark_math500(llm)

INFO 09-08 23:08:55 [utils.py:233] non-default args: {'max_model_len': 12000, 'gpu_memory_utilization': 0.85, 'generation_config': 'vllm', 'model': './merged'}
INFO 09-08 23:09:01 [model.py:549] Resolved architecture: Qwen3ForCausalLM
INFO 09-08 23:09:01 [model.py:1678] Using max model len 12000
INFO 09-08 23:09:01 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 09-08 23:09:01 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 09-08 23:09:02 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=4353) INFO 09-08 23:09:07 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='./merged', speculative_config=None, tokenizer='./merged', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.48it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.47it/s]
(EngineCore pid=4353) 


(EngineCore pid=4353) INFO 09-08 23:09:16 [default_loader.py:384] Loading weights took 0.90 seconds
(EngineCore pid=4353) INFO 09-08 23:09:16 [gpu_model_runner.py:4820] Model loading took 7.55 GiB memory and 5.949585 seconds
(EngineCore pid=4353) INFO 09-08 23:09:22 [backends.py:1051] Using cache directory: /root/.cache/vllm/torch_compile_cache/ef2f781800/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=4353) INFO 09-08 23:09:22 [backends.py:1111] Dynamo bytecode transform time: 5.90 s
(EngineCore pid=4353) INFO 09-08 23:09:28 [backends.py:372] Cache the graph of compile range (1, 16384) for later use
(EngineCore pid=4353) INFO 09-08 23:09:32 [backends.py:390] Compiling a graph for compile range (1, 16384) takes 9.72 s
(EngineCore pid=4353) INFO 09-08 23:09:33 [decorators.py:655] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/7e88aae8ea18aaf6c7eccffc47b1bdd281e4391a94080c6a783a1345826bf932/rank_0_0/model
(EngineCore pid=4353) INFO 09-08

(EngineCore pid=4353) 2026-09-08 23:09:41,732 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=4353) 2026-09-08 23:09:41,743 - INFO - autotuner.py:268 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 32.34it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 51/51 [00:01<00:00, 42.87it/s]


(EngineCore pid=4353) INFO 09-08 23:09:45 [gpu_model_runner.py:6046] Graph capturing finished in 3 secs, took 0.66 GiB
(EngineCore pid=4353) INFO 09-08 23:09:45 [gpu_worker.py:597] CUDA graph pool memory: 0.66 GiB (actual), 0.5 GiB (estimated), difference: 0.17 GiB (25.3%).
(EngineCore pid=4353) INFO 09-08 23:09:45 [core.py:283] init engine (profile, create kv cache, warmup model) took 28.69 seconds
(EngineCore pid=4353) INFO 09-08 23:09:46 [vllm.py:790] Asynchronous scheduling is enabled.


...

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Rendering conversations:   0%|          | 0/500 [00:00<?, ?it/s]

INFO 09-08 23:09:49 [hf.py:314] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts:   0%|          | 0/500 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

INFO 09-08 23:09:59 [loggers.py:259] Engine 000: Avg prompt throughput: 3560.6 tokens/s, Avg generation throughput: 18594.9 tokens/s, Running: 277 reqs, Waiting: 0 reqs, GPU KV cache usage: 46.6%, Prefix cache hit rate: 0.2%
INFO 09-08 23:10:09 [loggers.py:259] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 12759.5 tokens/s, Running: 124 reqs, Waiting: 0 reqs, GPU KV cache usage: 41.9%, Prefix cache hit rate: 0.2%
INFO 09-08 23:10:19 [loggers.py:259] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 7414.0 tokens/s, Running: 81 reqs, Waiting: 0 reqs, GPU KV cache usage: 42.3%, Prefix cache hit rate: 0.2%
INFO 09-08 23:10:29 [loggers.py:259] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 5293.4 tokens/s, Running: 63 reqs, Waiting: 0 reqs, GPU KV cache usage: 44.3%, Prefix cache hit rate: 0.2%
INFO 09-08 23:10:39 [loggers.py:259] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 

Training is **not** deterministic and vLLM itself has non-determinism even with greedy decoding.

Additionally, we reduce `max_tokens` to `11,000` rather than `16,384` to speed up evaluation time.

A score of 80%-86% is expected, this is well above the ~54% base model baseline.